# Problema 4 — Desempenho de algoritmos

Dois algoritmos, A e B, resolvem o mesmo problema. Seus tempos de execução, em milissegundos, são modelados por:

$$T_A(n) = 0{,}01\, n \log_2(n) \qquad\qquad T_B(n) = 0{,}08\, n + 20$$

**Objetivos:**
1. Determinar numericamente o tamanho de entrada $n$ em que os dois algoritmos têm o mesmo tempo.
2. Determinar, para $n$ **inteiro**, em quais tamanhos cada algoritmo é o mais rápido.
3. Comparar três métodos numéricos (bisseção, Newton-Raphson e secante), todos implementados no pacote `metodos_numericos`.

## 1. Formulação como $f(n) = 0$

Igualar os tempos, $T_A(n) = T_B(n)$, equivale a encontrar a raiz de

$$f(n) = T_A(n) - T_B(n) = 0{,}01\, n \log_2(n) - 0{,}08\, n - 20 = 0$$

- $f(n) > 0$: $T_A > T_B$ (B é mais rápido)
- $f(n) < 0$: $T_A < T_B$ (A é mais rápido)

Para o método de Newton precisamos da derivada. Como $\dfrac{d}{dn}\,[n \log_2 n] = \log_2 n + \dfrac{1}{\ln 2}$:

$$f'(n) = 0{,}01\left(\log_2 n + \frac{1}{\ln 2}\right) - 0{,}08$$

In [ ]:
import math
import time

import matplotlib.pyplot as plt

from metodos_numericos import bissecao, newton, secante

In [ ]:
def T_A(n):
    return 0.01 * n * math.log2(n)


def T_B(n):
    return 0.08 * n + 20


def f(n):
    return T_A(n) - T_B(n)


def df(n):
    return 0.01 * (math.log2(n) + 1 / math.log(2)) - 0.08


# Verificação da derivada analítica por diferença central
h = 1e-6
print(f"f'(1000) analítica : {df(1000):.9f}")
print(f"f'(1000) numérica  : {(f(1000 + h) - f(1000 - h)) / (2 * h):.9f}")

## 2. Visualização

Antes de aplicar os métodos, o gráfico ajuda a escolher um intervalo (bisseção) e chutes iniciais (Newton e secante).

In [ ]:
ns = range(2, 3000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ns, [T_A(n) for n in ns], color="tab:blue", label="$T_A(n)$")
ax1.plot(ns, [T_B(n) for n in ns], color="tab:red", label="$T_B(n)$")
ax1.set_xlabel("n")
ax1.set_ylabel("tempo (ms)")
ax1.set_title("Tempos de execução")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(ns, [f(n) for n in ns], color="tab:green")
ax2.axhline(0, color="gray", linewidth=0.8)
ax2.set_xlabel("n")
ax2.set_ylabel("f(n)")
ax2.set_title("f(n) = T_A(n) - T_B(n)")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

O gráfico mostra uma única troca de sinal, perto de $n \approx 1000$.

## 3. Parâmetros e critério de parada

| Parâmetro | Valor | Justificativa |
|---|---|---|
| Intervalo da bisseção | $[500, 2000]$ | $f(500) < 0$ e $f(2000) > 0$ (troca de sinal); $n > 0$ para o $\log_2$ existir |
| Chute do Newton | $x_0 = 1250$ | ponto médio do mesmo intervalo |
| Chutes da secante | $x_0 = 500,\ x_1 = 2000$ | extremos do mesmo intervalo |
| Tolerância | $\text{tol} = 10^{-6}$ | a mesma nos três métodos, para uma comparação justa |
| Máximo de iterações | 100 | trava de segurança |

**Critério de parada** (sempre sobre o erro em $n$):

- **Bisseção:** $(b-a)/2 < \text{tol}$, que **garante** erro menor que tol.
- **Newton e secante:** $|x_{k+1} - x_k| < \text{tol}$, que é uma **estimativa** do erro.

**Por que $10^{-6}$?** A raiz está perto de $1010$, então $10^{-6}$ dá precisão relativa de ~$10^{-9}$. Como $n$ deve ser inteiro e a raiz fica a ~0,017 do inteiro mais próximo, qualquer erro bem menor que isso já decide a questão. Com $10^{-6}$ há folga de quatro ordens de grandeza, e a bisseção gasta só $\lceil \log_2(1500/10^{-6}) \rceil = 31$ iterações.

**Por que não usar $|f(n)| < \text{tol}$?** Perto da raiz $f'(n) \approx 0{,}034$, então $|f| < \text{tol}$ aceitaria um $n$ com erro ~30 vezes maior que tol.

In [ ]:
TOL = 1e-6
MAX_ITER = 100

A, B = 500, 2000            # bisseção
X0_NEWTON = (A + B) / 2     # Newton
X0_SEC, X1_SEC = A, B       # secante

print(f"f({A})  = {f(A):+.4f}")
print(f"f({B}) = {f(B):+.4f}")

## 4. Aplicação dos três métodos

In [ ]:
res_b = bissecao(f, A, B, tol=TOL, max_iter=MAX_ITER)
res_n = newton(f, df, X0_NEWTON, tol=TOL, max_iter=MAX_ITER)
res_s = secante(f, X0_SEC, X1_SEC, tol=TOL, max_iter=MAX_ITER)

resultados = {"Bisseção": res_b, "Newton": res_n, "Secante": res_s}

for nome, r in resultados.items():
    print(f"{nome:9s}: raiz = {r.raiz:.10f} | f(raiz) = {r.f_raiz:+.2e} | "
          f"iterações = {r.iteracoes:2d} | convergiu = {r.convergiu}")
    print(f"           {r.mensagem}")

### Histórico da bisseção (primeiras e últimas iterações)

In [ ]:
print(f"{'k':>3} {'a':>12} {'b':>12} {'c':>12} {'f(c)':>11} {'erro':>10}")
hist = res_b.historico
for h_ in hist[:6] + [None] + hist[-3:]:
    if h_ is None:
        print("  ...")
        continue
    print(f"{h_['k']:>3} {h_['a']:>12.6f} {h_['b']:>12.6f} {h_['x']:>12.6f} "
          f"{h_['fx']:>+11.2e} {h_['erro']:>10.2e}")

## 5. Comparação dos métodos

Como referência para o erro real, usamos a própria bisseção com tolerância muito pequena ($10^{-12}$). O tempo é a média de várias execuções (inclui o custo de montar o histórico).

In [ ]:
raiz_ref = bissecao(f, A, B, tol=1e-12).raiz


def tempo_medio_us(func, repeticoes=2000):
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        func()
    return (time.perf_counter() - t0) / repeticoes * 1e6


chamadas = {
    "Bisseção": lambda: bissecao(f, A, B, tol=TOL),
    "Newton": lambda: newton(f, df, X0_NEWTON, tol=TOL),
    "Secante": lambda: secante(f, X0_SEC, X1_SEC, tol=TOL),
}

print(f"Referência: n* = {raiz_ref:.10f}\n")
print(f"{'Método':<10} {'raiz':>16} {'erro real':>11} {'|f(raiz)|':>11} {'iter.':>6} {'tempo (µs)':>11}")
for nome, r in resultados.items():
    print(f"{nome:<10} {r.raiz:>16.10f} {abs(r.raiz - raiz_ref):>11.2e} "
          f"{abs(r.f_raiz):>11.2e} {r.iteracoes:>6d} {tempo_medio_us(chamadas[nome]):>11.1f}")

### Convergência do erro estimado

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for nome, r in resultados.items():
    ks = [h_["k"] for h_ in r.historico]
    erros = [h_["erro"] for h_ in r.historico]
    ax.semilogy(ks, erros, "o-", markersize=3, label=nome)
ax.axhline(TOL, color="gray", linestyle="--", linewidth=0.8, label="tol")
ax.set_xlabel("iteração k")
ax.set_ylabel("erro estimado")
ax.set_title("Convergência dos métodos")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()

### Sensibilidade à tolerância

Número de iterações necessárias para diferentes tolerâncias (erro real entre parênteses).

In [ ]:
print(f"{'tol':>8} | {'Bisseção':>18} | {'Newton':>18} | {'Secante':>18}")
for tol in [1e-3, 1e-6, 1e-10]:
    linha = []
    for r in (bissecao(f, A, B, tol=tol),
              newton(f, df, X0_NEWTON, tol=tol),
              secante(f, X0_SEC, X1_SEC, tol=tol)):
        linha.append(f"{r.iteracoes:>3d} ({abs(r.raiz - raiz_ref):.1e})")
    print(f"{tol:>8.0e} | " + " | ".join(f"{s:>18}" for s in linha))

## 6. Interpretação final ($n$ inteiro)

A raiz real está entre dois inteiros consecutivos. Como $n$ deve ser inteiro, avaliamos $T_A$ e $T_B$ diretamente nos inteiros vizinhos, sem depender da precisão do método.

In [ ]:
n_baixo = math.floor(res_n.raiz)
n_alto = math.ceil(res_n.raiz)
print(f"Raiz real: {res_n.raiz:.6f}  ->  vizinhos inteiros: {n_baixo} e {n_alto}\n")

print(f"{'n':>6} {'T_A(n)':>11} {'T_B(n)':>11} {'f(n)':>10}  mais rápido")
for n in range(n_baixo - 1, n_alto + 2):
    fn = f(n)
    vence = "A" if fn < 0 else ("B" if fn > 0 else "empate")
    print(f"{n:>6} {T_A(n):>11.5f} {T_B(n):>11.5f} {fn:>+10.5f}  {vence}")

In [ ]:
# Confirmação por varredura de todos os inteiros do intervalo do gráfico
faixa_A = [n for n in range(1, 3000) if T_A(n) < T_B(n)]
faixa_B = [n for n in range(1, 3000) if T_A(n) > T_B(n)]
empates = [n for n in range(1, 3000) if T_A(n) == T_B(n)]

print(f"A mais rápido: n de {faixa_A[0]} a {faixa_A[-1]}")
print(f"B mais rápido: n de {faixa_B[0]} a {faixa_B[-1]} (limite da varredura)")
print(f"Empates entre inteiros: {empates if empates else 'nenhum'}")

**Resposta:**

- O tamanho de entrada em que os tempos são iguais é $n^* \approx 1010{,}0172$ (não é inteiro).
- Para **$1 \le n \le 1010$**, o **algoritmo A** tem o menor tempo de execução.
- Para **$n \ge 1011$**, o **algoritmo B** tem o menor tempo de execução.
- Não existe inteiro com tempos exatamente iguais: em $n = 1010$ a diferença é de apenas $0{,}0006$ ms, a favor de A.

## 7. Conclusões

- Os três métodos convergem para a mesma raiz, $n^* \approx 1010{,}0172$, concordando em pelo menos 6 casas decimais.
- **Bisseção:** convergência garantida e erro controlável, mas linear e lenta (31 iterações para $10^{-6}$). Exige um intervalo com troca de sinal.
- **Newton:** convergência quadrática (4 iterações), mas exige a derivada e um bom chute inicial. Testes fora deste roteiro mostraram que, de $x_0 = 50$, o método sai do domínio do logaritmo e falha, e o pacote reporta isso com `convergiu = False`.
- **Secante:** convergência superlinear (6 iterações) sem precisar da derivada, com custo por iteração menor que o de Newton (uma avaliação de $f$).
- O critério de parada sobre o erro em $n$, com a mesma tolerância nos três métodos, permitiu uma comparação justa.